In [ ]:
import dotenv
dotenv.load_dotenv()

In [ ]:
import os
QWEN_ENDPOINT = os.getenv('QWEN_ENDPOINT')
QWEN_MODEL = 'Qwen/Qwen-Image'

In [ ]:
import io
import base64
import random
import string

from pathlib import Path
from datetime import datetime

import requests
from PIL import Image


def base64_decode(b64: str) -> Image.Image:
    png = base64.b64decode(b64)
    return Image.open(io.BytesIO(png))


def save_image(image: Image, filename: str) -> str:
    path = Path(filename)
    path.parent.mkdir(parents=True, exist_ok=True)
    image.save(path)
    return str(path)


def generate_random_filename():
    ts = datetime.now().strftime("%Y%m%d-%H%M%S")
    rand = "".join(random.choices(string.ascii_lowercase + string.digits, k=3))
    return f"out/quickstart/qwen_{ts}_{rand}.png"


class ImageGenerator:

    def __init__(self, endpoint: str, model: str):
        self.endpoint = endpoint
        self.model = model 
    
    def generate_image(self, prompt: str) -> Image.Image:
        request_data = {
            "model": self.model,
            "prompt": prompt,
            "size": "1024x1024",
            "seed": 7,
            "response_format": "b64_json",
            "num_inference_steps": 20,
        }

        response = requests.post(
            f"{self.endpoint}/v1/images/generations",
            json=request_data,
        )
        response.raise_for_status()
        response_json = response.json()

        b64 = response_json["data"][0]["b64_json"]
        return base64_decode(b64)

    def generate_image_tool(self, prompt: str) -> str:
        """
        Generate an illustration and save it, return the path.
        """
        png = self.generate_image(prompt)
        filename = generate_random_filename()
        return save_image(png, filename)



In [ ]:
qwen_generator = ImageGenerator(QWEN_ENDPOINT, QWEN_MODEL)
qwen_generator.generate_image('telescope')

## Run the agent on a kid profile

In [ ]:
instructions = """

You are Santa's creative media agent.

Given a child's profile and wishlist:

1. Choose one hero gift inspired by the wishlist that is magical and
   age-appropriate.
2. Write a detailed Christmas storybook illustration prompt:
   - Centered composition featuring the hero gift glowing softly on snow
     under a starry night sky
   - Surrounded by a small decorated Christmas tree, wrapped presents,
     twinkling lights, gentle falling snow
   - Warm, vibrant colors, soft lighting, whimsical children's-book art style, 
     clean shapes, cinematic depth
   - No people, no faces, no animals with faces, no text, no letters, no signatures,
     no watermarks
3. Call `generate_image_tool` once with that illustration prompt.
4. Return a warm 2-sentence holiday wish addressing the child by name and ending with
   "Merry Christmas!", followed on a new line by exactly one HTML image tag showing
   the saved file:
   <img src="PATH_FROM_TOOL" width="420">
   Replace PATH_FROM_TOOL with the exact path returned by `generate_image_tool`
   (e.g. <img src="out/quickstart/qwen_20261103-120000_abc.png" width="420">).
   Output raw HTML only: no code fences, no markdown.

""".strip()


In [ ]:
from pydantic_ai import Agent

agent = Agent(
    model="openai:gpt-6-luna",
    instructions=instructions,
    tools=[qwen_generator.generate_image_tool],
)

In [ ]:
prompt_template = """
Create a personalized Christmas illustration for {name} 
age {age} with whislist {wishlist}
""".strip()

In [ ]:

kid = {"name": "Emma", "age": 7, "wishlist": ["a telescope"]}

prompt = prompt_template.format(
    name=kid['name'],
    age=kid['age'],
    wishlist=",".join(kid['wishlist'])
)

print(prompt)

In [ ]:

result = await agent.run(prompt)
print(result.output)


In [ ]:
from IPython.display import HTML, display

In [ ]:
display(HTML(result.output))

In [ ]:
SANA_ENDPOINT = os.getenv('SANA_ENDPOINT')
SANA_MODEL = "Efficient-Large-Model/Sana_1600M_1024px_diffusers"

In [ ]:
sana_generator = ImageGenerator(SANA_ENDPOINT, SANA_MODEL)
sana_generator.generate_image('telescope')

In [ ]:
agent = Agent(
    model="openai:gpt-6-luna",
    instructions=instructions,
    tools=[sana_generator.generate_image_tool],
)

In [ ]:

result = await agent.run(prompt)
display(HTML(result.output))